In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Dict, List

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 11 — FUNCTION 3 (Clustering Techniques v1)
# What changed vs Week 10:
#  1) Clustering lens: KMeans on standardized X to detect “regions”
#  2) k chosen via silhouette score (k=2..6) for a stable partition
#  3) Candidate pools:
#       - Global Sobol (coverage)
#       - Top-cluster centroid + best-in-cluster (centroid trend tightening)
#       - Between-cluster “bridge” samples (boundary probing)
#  4) Score remains decomposed (EI/UCB/repulsion/boundary) + cluster bonus
#  5) Full transparency: prints cluster summary + which cluster x_next targets
# ============================================================

DEVICE = torch.device("cpu")  # set "cuda" if available

# ---------------------------------------------------------
# 1. Data: 3D inputs and 1D outputs (maximisation)
# ---------------------------------------------------------
X_raw = np.array([
    [0.17152521, 0.34391687, 0.2487372],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.6970324 , 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.2462953 ],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],
    [1.065994  , 1.041359  , 1.090881  ],  # historical out-of-bounds (kept; clipped)
    [0.403482  , 0.38217   , 0.489363  ],
    [3.98350e-01, 1.00000e-06, 5.43642e-01],
    [0.962851  , 0.987386  , 0.040875  ],
    [0.504564  , 0.348726  , 0.601264  ],
    [0.265159  , 0.286931  , 0.413777  ],
    [0.403756  , 0.381706  , 0.489738  ],
    [0.359927  , 0.175969  , 0.720956  ],
    [0.846635  , 0.969038  , 0.008713  ],
    [0.952304, 0.517975, 0.686149]
], dtype=float)

y_raw = np.array([
    -0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758,
    -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
    -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
    -0.769427956661122, -0.03310307977430594, -0.09333459499358941, -0.07627377706316849, -0.05678719487656195,
    -0.03492633073917894, -0.009136026447950633, -0.14476549871155003, -0.11906253814017263, -0.1409808967165733
], dtype=float)


# ---------------------------------------------------------
# Utility: bounds + rounding for submission
# ---------------------------------------------------------
def clip_to_bounds(X: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    return np.clip(X, lower, upper)

def round6(x: np.ndarray) -> np.ndarray:
    return np.round(x.astype(float), 6)

def as_fixed6_list(x: np.ndarray) -> str:
    return f"[{x[0]:.6f}, {x[1]:.6f}, {x[2]:.6f}]"

def frac_out_of_bounds(X: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> float:
    bad = ((X < lower) | (X > upper)).any(axis=1)
    return float(np.mean(bad))


# ---------------------------------------------------------
# 2. PyTorch MLP model
# ---------------------------------------------------------
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes=(64, 64), dropout=0.15):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def _train_mlp(
    Xs: np.ndarray,
    ys: np.ndarray,
    hidden: Tuple[int, ...],
    dropout: float,
    lr: float,
    weight_decay: float,
    n_epochs: int,
    seed: int,
    tol: float = 1e-6,
    patience: int = 80,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLPRegressorTorch(
        input_dim=Xs.shape[1],
        hidden_sizes=hidden,
        dropout=dropout
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
    y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

    best_loss = float("inf")
    bad = 0

    for _ in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = criterion(preds, y_tensor)
        loss.backward()
        optimizer.step()

        l = float(loss.item())
        if best_loss - l > tol:
            best_loss = l
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return model


# ---------------------------------------------------------
# 3. MC Dropout surrogate (robust y scaling)
# ---------------------------------------------------------
@dataclass
class MCDropoutSurrogate:
    hidden_layer_sizes: Tuple[int, ...] = (64, 64)
    dropout: float = 0.15
    n_epochs: int = 1200
    lr: float = 1e-3
    weight_decay: float = 1e-6
    random_state: int = 0
    n_mc_samples: int = 128
    robust_y: bool = True

    def __post_init__(self):
        self.model = None
        self.x_scaler = StandardScaler()
        self.y_scaler = RobustScaler() if self.robust_y else StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.fit_transform(Xc)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        self.model = _train_mlp(
            Xs, ys,
            hidden=self.hidden_layer_sizes,
            dropout=self.dropout,
            lr=self.lr,
            weight_decay=self.weight_decay,
            n_epochs=self.n_epochs,
            seed=self.random_state,
        )

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.transform(Xc)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        # Keep dropout ON at inference for epistemic uncertainty
        self.model.train()

        preds_scaled_mc = []
        with torch.no_grad():
            for _ in range(self.n_mc_samples):
                preds_scaled_mc.append(self.model(X_tensor).cpu().numpy().ravel())

        preds_scaled_mc = np.stack(preds_scaled_mc, axis=0)
        mean_scaled = preds_scaled_mc.mean(axis=0)
        std_scaled = preds_scaled_mc.std(axis=0)

        scale_y = float(self.y_scaler.scale_[0])
        center_y = float(self.y_scaler.center_[0])

        mean = mean_scaled * scale_y + center_y
        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std

    def local_input_gradients(self, x: np.ndarray) -> np.ndarray:
        """
        Gradient of predicted mean (dropout OFF) w.r.t. ORIGINAL inputs.
        Returns a (3,) vector in original input units.
        """
        assert self.model is not None
        x = np.asarray(x, float).reshape(1, -1)
        self.model.eval()

        xs = self.x_scaler.transform(x).astype(np.float32)
        xt = torch.tensor(xs, device=DEVICE, requires_grad=True)

        y_scaled = self.model(xt).view(1)
        scale_y = float(self.y_scaler.scale_[0])
        y_orig = y_scaled * scale_y
        y_orig.backward()

        grad_xs = xt.grad.detach().cpu().numpy().reshape(-1)
        x_std = self.x_scaler.scale_.reshape(-1)
        grad_x = grad_xs / np.maximum(x_std, 1e-12)
        return grad_x


# ---------------------------------------------------------
# 3b. Ensemble wrapper
# ---------------------------------------------------------
@dataclass
class EnsembleSurrogate:
    members: List[MCDropoutSurrogate]

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        for m in self.members:
            m.fit(X, y, lower, upper)

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        mus, sigs = [], []
        for m in self.members:
            mu, sig = m.predict(X, lower, upper, return_std=True)
            mus.append(mu)
            sigs.append(sig)

        mus = np.stack(mus, axis=0)
        sigs = np.stack(sigs, axis=0)

        mu_ens = mus.mean(axis=0)
        var_within = (sigs ** 2).mean(axis=0)
        var_between = mus.var(axis=0)
        sig_ens = np.sqrt(np.maximum(var_within + var_between, 1e-12))

        if not return_std:
            return mu_ens
        return mu_ens, sig_ens

    def local_gradients(self, x: np.ndarray) -> Dict:
        grads = np.stack([m.local_input_gradients(x) for m in self.members], axis=0)
        return {
            "grad_mean": grads.mean(axis=0),
            "grad_std": grads.std(axis=0),
            "all_grads": grads
        }


# ---------------------------------------------------------
# 4. Acquisition helpers
# ---------------------------------------------------------
def acquisition_pi_ei(mu: np.ndarray, sigma: np.ndarray, y_best: float, xi: float = 0.0):
    sigma = np.maximum(sigma, 1e-9)
    gamma = (mu - y_best - xi) / sigma
    pi = norm.cdf(gamma)
    ei = (mu - y_best - xi) * pi + sigma * norm.pdf(gamma)
    return pi, np.maximum(ei, 0.0)


# ---------------------------------------------------------
# 5. Week 11: clustering utilities
# ---------------------------------------------------------
def choose_k_by_silhouette(Xs: np.ndarray, k_min: int = 2, k_max: int = 6, seed: int = 0) -> Tuple[int, float]:
    best_k = k_min
    best_s = -1.0
    max_k = min(k_max, len(Xs) - 1)
    for k in range(k_min, max_k + 1):
        km = KMeans(n_clusters=k, random_state=seed, n_init=25)
        labels = km.fit_predict(Xs)
        if len(set(labels)) < 2:
            continue
        s = float(silhouette_score(Xs, labels))
        if s > best_s:
            best_s = s
            best_k = k
    return best_k, best_s


# ---------------------------------------------------------
# 6. Week 11 proposer: global + cluster tightening + boundary probing
# ---------------------------------------------------------
def propose_next_point_week11(
    surrogate: EnsembleSurrogate,
    X_obs: np.ndarray,
    y_obs: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 123,

    # Candidate pool sizes (keep similar scale to W10; adjust if runtime-heavy)
    n_sobol: int = 60_000,
    n_cluster: int = 40_000,
    n_boundary: int = 15_000,

    # Constraints
    min_dist: float = 0.010,
    repulse_len: float = 0.050,
    repulse_w: float = 0.30,

    # Acquisition mix
    xi_base: float = 0.0010,
    alpha_ucb: float = 0.35,

    # Edge safety
    bound_margin: float = 0.006,

    # Clustering controls
    k_max: int = 6,
    cluster_bonus_w: float = 0.015,
    base_sigma: float = 0.060,

    enforce_rounded_nonduplicate: bool = True,
) -> Dict:
    rng = np.random.RandomState(random_state)
    lower, upper = np.asarray(bounds[0], float), np.asarray(bounds[1], float)

    Xc = clip_to_bounds(X_obs, lower, upper)

    best_idx = int(np.argmax(y_obs))
    y_best = float(y_obs[best_idx])

    n = len(y_obs)
    d = Xc.shape[1]

    # --- clustering lens (standardized space for distance comparability) ---
    x_scaler = StandardScaler()
    Xs = x_scaler.fit_transform(Xc)

    k, sil = choose_k_by_silhouette(Xs, k_min=2, k_max=k_max, seed=random_state)
    km = KMeans(n_clusters=k, random_state=random_state, n_init=50)
    labels = km.fit_predict(Xs)

    centroids_s = km.cluster_centers_
    centroids = x_scaler.inverse_transform(centroids_s)

    # Rank clusters by (best y, mean y)
    cluster_stats = []
    for c in range(k):
        idx = np.where(labels == c)[0]
        cluster_stats.append((
            int(c),
            float(np.max(y_obs[idx])),
            float(np.mean(y_obs[idx])),
            int(len(idx))
        ))
    cluster_stats.sort(key=lambda t: (t[1], t[2]), reverse=True)

    top_clusters = [c for (c, _, _, _) in cluster_stats[:min(3, k)]]

    # exploration decays gently
    xi = float(xi_base + 0.008 / np.sqrt(max(n, 1)))
    beta = float(0.55 * np.sqrt(np.log(n + 2.0)))

    # --- Candidate generation ---
    # (1) Global Sobol
    sob = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=int(random_state))
    X_sob = sob.draw(n_sobol).cpu().numpy()
    X_sob = lower + (upper - lower) * X_sob

    # (2) Cluster tightening: around centroid + best-in-cluster
    X_cl = []
    for c in top_clusters:
        idx = np.where(labels == c)[0]
        best_in_c = idx[int(np.argmax(y_obs[idx]))]
        x_best_c = Xc[best_in_c]

        # “boundary tightening”: smaller sigma when the cluster is tighter in standardized space
        spread = float(np.mean(np.linalg.norm(Xs[idx] - centroids_s[c], axis=1))) if len(idx) > 1 else 0.6
        sigma = base_sigma * (0.6 + 0.7 / (1.0 + spread))

        m = n_cluster // len(top_clusters)
        X1 = centroids[c] + rng.normal(0.0, sigma, size=(m // 2, d))
        X2 = x_best_c   + rng.normal(0.0, sigma * 0.6, size=(m - m // 2, d))
        X_cl.append(np.vstack([X1, X2]))

    X_cl = np.vstack(X_cl) if len(X_cl) else np.empty((0, d))

    # (3) Between-cluster boundary probing: interpolate between top-2 centroids in standardized space
    X_bd = np.empty((0, d))
    if len(top_clusters) >= 2:
        c1, c2 = top_clusters[0], top_clusters[1]
        t = rng.uniform(0.2, 0.8, size=(n_boundary, 1))
        line = (1 - t) * centroids_s[c1] + t * centroids_s[c2]
        line = line + rng.normal(0.0, 0.15, size=line.shape)
        X_bd = x_scaler.inverse_transform(line)

    Xcand = clip_to_bounds(np.vstack([X_sob, X_cl, X_bd]), lower, upper)

    # --- Surrogate predictions ---
    mu, sig = surrogate.predict(Xcand, lower, upper, return_std=True)
    pi, ei = acquisition_pi_ei(mu, sig, y_best=y_best, xi=xi)

    # --- Distances to observed points (hard min distance + soft repulsion) ---
    dmat = np.linalg.norm(Xcand[:, None, :] - Xc[None, :, :], axis=2)
    dmin = dmat.min(axis=1)
    ok = dmin >= min_dist
    repulse_pen = np.exp(-(dmin ** 2) / (2.0 * repulse_len ** 2))

    # --- Bound margin penalty (avoid hugging edges) ---
    dist_to_lower = (Xcand - lower)
    dist_to_upper = (upper - Xcand)
    bound_close = np.minimum(dist_to_lower, dist_to_upper).min(axis=1)
    bound_pen = np.exp(- (bound_close / max(bound_margin, 1e-9))**2 )

    # --- EI/UCB components ---
    ucb_impr = np.maximum(mu + beta * sig - y_best, 0.0)

    ei_max = float(np.max(ei[np.isfinite(ei)]) if np.isfinite(ei).any() else 1.0)
    ei_norm = ei / max(ei_max, 1e-12)

    comp_ei = (1.0 - alpha_ucb) * ei_norm
    comp_ucb = alpha_ucb * ucb_impr
    comp_repulse = repulse_w * repulse_pen
    comp_bound = 0.10 * bound_pen

    # --- Cluster proximity bonus (in standardized space) ---
    Xcand_s = x_scaler.transform(Xcand)
    bonus = np.zeros(len(Xcand), float)
    for r, c in enumerate(top_clusters):
        ds = np.linalg.norm(Xcand_s - centroids_s[c], axis=1)
        bonus += cluster_bonus_w * np.exp(-(ds ** 2) / (2.0 * (1.0 + 0.5 * r) ** 2))

    # Final score (decomposed)
    score = comp_ei + comp_ucb + bonus - comp_repulse - comp_bound
    score = np.where(ok, score, -np.inf)

    # Relax once if too constrained
    if not np.isfinite(score).any():
        ok2 = dmin >= (0.5 * min_dist)
        score = np.where(ok2, comp_ei + comp_ucb + bonus - comp_repulse - comp_bound, -np.inf)

    next_idx = int(np.argmax(score))
    next_x = Xcand[next_idx].copy()
    next_x_6 = round6(next_x)

    # Rounded non-duplicate enforcement
    pick_mode = "Week11: Sobol + cluster tightening + boundary probing"
    if enforce_rounded_nonduplicate:
        X_obs_6 = round6(clip_to_bounds(X_obs, lower, upper))
        seen = set(map(tuple, X_obs_6))
        if tuple(next_x_6) in seen:
            order = np.argsort(score)[::-1]
            for j in order[:15000]:
                cand6 = round6(Xcand[j])
                if tuple(cand6) not in seen:
                    next_idx = int(j)
                    next_x = Xcand[next_idx].copy()
                    next_x_6 = cand6
                    pick_mode += " + nondup-fallback"
                    break

    # which cluster does x_next fall into? (assign by nearest centroid in standardized space)
    xn_s = x_scaler.transform(next_x_6.reshape(1, -1))
    d_to_cent = np.linalg.norm(xn_s - centroids_s, axis=1)
    target_cluster = int(np.argmin(d_to_cent))

    # nearest neighbors for audit
    dists = np.linalg.norm(Xc - next_x, axis=1)
    nn = np.argsort(dists)[:3]

    decomp = dict(
        score=float(score[next_idx]),
        ei=float(ei[next_idx]),
        ei_norm=float(ei_norm[next_idx]),
        ucb_impr=float(ucb_impr[next_idx]),
        comp_ei=float(comp_ei[next_idx]),
        comp_ucb=float(comp_ucb[next_idx]),
        comp_repulse=float(comp_repulse[next_idx]),
        comp_bound=float(comp_bound[next_idx]),
        cluster_bonus=float(bonus[next_idx]),
        pi=float(pi[next_idx]),
        dmin=float(dmin[next_idx]),
        bound_close=float(bound_close[next_idx]),
        xi=float(xi),
        beta=float(beta),
        alpha_ucb=float(alpha_ucb),
        k=int(k),
        silhouette=float(sil),
        target_cluster=int(target_cluster),
        top_clusters=[int(c) for c in top_clusters],
        pick_mode=pick_mode
    )

    reasoning = [
        "WEEK 11 CLUSTERING NOTES (Function 3)",
        "",
        "How past patterns shaped this submission:",
        f"  - Best observed y={y_best:.6f} suggests a promising region; we tighten around that region’s cluster.",
        "  - Earlier scattered low returns motivate using clusters to separate signal (promising region) from noise.",
        "",
        "Clusters / recurring regions (what we found):",
        f"  - KMeans on standardized X with silhouette-selected k={k} (silhouette={sil:.3f}).",
        f"  - Top clusters ranked by (best y, mean y): {cluster_stats[:min(3,len(cluster_stats))]}",
        f"  - x_next targets cluster={target_cluster} (nearest centroid in standardized space).",
        "",
        "Less effective strategies / adjustments:",
        "  - Purely global exploration often lands in low-value regions → keep Sobol pool but add cluster tightening.",
        "  - Edge-hugging / out-of-bounds history → keep bound margin penalty + clipping.",
        "",
        "How this parallels clustering separating signal from noise:",
        "  - Treats regions as groups: exploit the group with the best outcomes (signal).",
        "  - Still probes between groups (boundary candidates) to avoid missing a better cluster.",
        "",
        "If plotted, what trends/groupings might appear:",
        "  - A denser cloud near the best-performing region (cluster tightening).",
        "  - A thinner ‘bridge’ of samples between the top two clusters (boundary probing).",
        "",
        "Selection summary:",
        f"  - Mode: {pick_mode}",
        f"  - Score = comp_ei + comp_ucb + cluster_bonus - comp_repulse - comp_bound",
        f"  - xi={xi:.6f}, beta={beta:.4f}, alpha_ucb={alpha_ucb}",
        "",
        "Chosen point (audit):",
        f"  - x_next_6dp={next_x_6}",
        f"  - μ={float(mu[next_idx]):.6f}, σ={float(sig[next_idx]):.6f}, PI={float(pi[next_idx]):.4f}, EI={float(ei[next_idx]):.6f}",
        f"  - Decomp: comp_ei={decomp['comp_ei']:.6f}, comp_ucb={decomp['comp_ucb']:.6f}, "
        f"bonus={decomp['cluster_bonus']:.6f}, repulse={decomp['comp_repulse']:.6f}, bound={decomp['comp_bound']:.6f}, score={decomp['score']:.6f}",
        f"  - dmin_to_data={decomp['dmin']:.6f}, bound_close={decomp['bound_close']:.6f}",
        "",
        "Nearest tested points:",
    ]
    for i, idx in enumerate(nn, 1):
        reasoning.append(
            f"  #{i}: x={round6(X_obs[idx])}, y={float(y_obs[idx]):.6f}, dist={float(dists[idx]):.6f}"
        )

    return dict(
        next_x=next_x_6,
        next_x_raw=next_x,
        pred_mean=float(mu[next_idx]),
        pred_std=float(sig[next_idx]),
        y_best=y_best,
        x_best=round6(X_obs[best_idx]),
        decomp=decomp,
        cluster_stats=cluster_stats,
        centroids=centroids,
        reasoning="\n".join(reasoning),
    )


# ---------------------------------------------------------
# 7. Hyperparameter tuning (kept as-is from Week 10)
# ---------------------------------------------------------
def cv_mse_score(
    config: Dict,
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    k: int = 5,
    seed: int = 0
) -> float:
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    mses = []
    lower, upper = bounds

    for tr_idx, va_idx in kf.split(X):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        surr = MCDropoutSurrogate(
            hidden_layer_sizes=config["hidden"],
            dropout=config["dropout"],
            n_epochs=config["epochs"],
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            random_state=seed,
            n_mc_samples=config["n_mc_samples"],
            robust_y=True
        )

        surr.fit(Xtr, ytr, lower, upper)
        preds = surr.predict(Xva, lower, upper)
        mses.append(np.mean((preds - yva) ** 2))

    return float(np.mean(mses))

def tune_hyperparameters(
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 8
) -> Dict:
    rng = np.random.RandomState(random_state)

    hidden_options = [(64, 32), (64, 64), (128, 64)]
    dropout_options = [0.10, 0.15, 0.20]
    lr_options = [7e-4, 1e-3, 2e-3]
    wd_options = [0.0, 1e-6, 1e-5]

    n_initial = 10
    configs = []
    for _ in range(n_initial):
        cfg = dict(
            hidden=hidden_options[rng.randint(len(hidden_options))],
            dropout=float(dropout_options[rng.randint(len(dropout_options))]),
            lr=float(lr_options[rng.randint(len(lr_options))]),
            weight_decay=float(wd_options[rng.randint(len(wd_options))]),
            n_mc_samples=int([96, 128][rng.randint(2)]),
        )
        configs.append(cfg)

    stage_epochs = [450, 1050]
    keep_fracs = [0.5, 0.4]

    best_overall = None

    for stage, (epochs, keep_frac) in enumerate(zip(stage_epochs, keep_fracs), start=1):
        scored = []
        for cfg in configs:
            cfg_stage = dict(cfg)
            cfg_stage["epochs"] = epochs
            mse = cv_mse_score(cfg_stage, X, y, bounds=bounds, k=5, seed=0)
            scored.append((mse, cfg_stage))

        scored.sort(key=lambda t: t[0])
        if best_overall is None or scored[0][0] < best_overall[0]:
            best_overall = scored[0]

        k_keep = max(4, int(len(scored) * keep_frac))
        configs = [cfg for _, cfg in scored[:k_keep]]

        print(f"\n--- TUNING STAGE {stage} (W11) ---")
        print(f"epochs={epochs}, kept={k_keep}/{len(scored)}")
        print(f"best CV-MSE so far: {best_overall[0]:.6f}")
        print(f"best config so far: {best_overall[1]}")

    return dict(best_cv_mse=best_overall[0], best_config=best_overall[1])


# ---------------------------------------------------------
# 8. Main
# ---------------------------------------------------------
def main():
    np.random.seed(0)
    torch.manual_seed(0)

    bounds = (np.zeros(3), np.ones(3))
    lower, upper = bounds

    print("================================================")
    print("WEEK 11 FUNCTION 3 — CLUSTERING PRE-CHECKS")
    print("================================================")
    print(f"Points (n): {len(X_raw)}")
    print(f"Frac out-of-bounds in X_raw (before clip): {frac_out_of_bounds(X_raw, lower, upper):.3f}")

    # Tune hyperparameters
    tuning = tune_hyperparameters(X_raw, y_raw, bounds=bounds, random_state=8)
    best_cfg = tuning["best_config"]

    # Fit ensemble
    seeds = [0, 11, 29]  # fixed for determinism
    members = []
    for s in seeds:
        members.append(
            MCDropoutSurrogate(
                hidden_layer_sizes=best_cfg["hidden"],
                dropout=best_cfg["dropout"],
                n_epochs=best_cfg["epochs"],
                lr=best_cfg["lr"],
                weight_decay=best_cfg["weight_decay"],
                random_state=s,
                n_mc_samples=best_cfg["n_mc_samples"],
                robust_y=True
            )
        )

    surrogate = EnsembleSurrogate(members=members)
    surrogate.fit(X_raw, y_raw, lower, upper)

    # Current best
    best_idx = int(np.argmax(y_raw))
    current_best_x = round6(clip_to_bounds(X_raw[best_idx], lower, upper))
    current_best_y = float(y_raw[best_idx])

    # Propose next query (Week 11)
    suggestion = propose_next_point_week11(
        surrogate=surrogate,
        X_obs=X_raw,
        y_obs=y_raw,
        bounds=bounds,
        random_state=123,
        n_sobol=60_000,
        n_cluster=40_000,
        n_boundary=15_000,
        min_dist=0.010,
        repulse_len=0.050,
        repulse_w=0.30,
        xi_base=0.0010,
        alpha_ucb=0.35,
        bound_margin=0.006,
        k_max=6,
        cluster_bonus_w=0.015,
        base_sigma=0.060,
        enforce_rounded_nonduplicate=True
    )

    x_next = suggestion["next_x"]

    # Interpretability: local gradients at chosen point
    grads = surrogate.local_gradients(x_next)
    g_mean = grads["grad_mean"]
    g_std = grads["grad_std"]
    g_abs = np.abs(g_mean)
    g_norm = g_abs / max(float(g_abs.sum()), 1e-12)

    print("\n================================================")
    print("WEEK 11 FUNCTION 3 — NEXT POINT (6 DECIMALS)")
    print("================================================")
    print("Surrogate: ensemble(mc_dropout) + robust y scaling")
    print(f"Best CV-MSE (lower is better): {tuning['best_cv_mse']:.6f}")
    print("Best tuned hyperparameters:")
    print(f"  hidden: {best_cfg['hidden']}")
    print(f"  dropout: {best_cfg['dropout']}")
    print(f"  lr: {best_cfg['lr']}")
    print(f"  weight_decay: {best_cfg['weight_decay']}")
    print(f"  n_mc_samples: {best_cfg['n_mc_samples']}")
    print(f"  epochs: {best_cfg['epochs']}")
    print(f"Ensemble seeds: {seeds}")

    print("\n================================================")
    print("CURRENT BEST OBSERVED")
    print("================================================")
    print(f"x_best = {as_fixed6_list(current_best_x)}, y_best = {current_best_y:.6f}")

    print("\n================================================")
    print("RECOMMENDED NEXT POINT (rounded to 6 decimals)")
    print("================================================")
    print(f"x_next     = [{x_next[0]:.6f}, {x_next[1]:.6f}, {x_next[2]:.6f}]")
    print(f"μ(x_next)  = {suggestion['pred_mean']:.6f}")
    print(f"σ(x_next)  = {suggestion['pred_std']:.6f}")

    print("\n================================================")
    print("CLUSTERING SUMMARY (TRANSPARENCY)")
    print("================================================")
    d = suggestion["decomp"]
    print(f"k (silhouette-selected) = {d['k']} | silhouette={d['silhouette']:.3f}")
    print(f"top_clusters (ranked by best/mean y) = {d['top_clusters']}")
    print(f"x_next targets cluster = {d['target_cluster']}")

    print("\n================================================")
    print("TRANSPARENT SCORE DECOMPOSITION @ x_next")
    print("================================================")
    print(f"score       = {d['score']:.6f}")
    print(f"comp_ei     = {d['comp_ei']:.6f}  (EI_norm={d['ei_norm']:.6f}, xi={d['xi']:.6f})")
    print(f"comp_ucb    = {d['comp_ucb']:.6f}  (UCB_impr={d['ucb_impr']:.6f}, beta={d['beta']:.4f})")
    print(f"cluster_bonus= {d['cluster_bonus']:.6f}")
    print(f"comp_repulse= {d['comp_repulse']:.6f}  (dmin={d['dmin']:.6f})")
    print(f"comp_bound  = {d['comp_bound']:.6f}  (bound_close={d['bound_close']:.6f}, margin={0.006:.3f})")

    print("\n================================================")
    print("INTERPRETABILITY: LOCAL SENSITIVITY (GRADIENTS)")
    print("================================================")
    print("Gradient of predicted mean wrt inputs (dropout OFF for stability).")
    print(f"grad_mean = [{g_mean[0]:.6f}, {g_mean[1]:.6f}, {g_mean[2]:.6f}]")
    print(f"grad_std  = [{g_std[0]:.6f}, {g_std[1]:.6f}, {g_std[2]:.6f}]")
    print("Relative absolute importance (|grad| normalized):")
    print(f"  dim1: {g_norm[0]:.3f}, dim2: {g_norm[1]:.3f}, dim3: {g_norm[2]:.3f}")

    print("\n================================================")
    print("REASONING (AUDIT TRAIL)")
    print("================================================")
    print(suggestion["reasoning"])


if __name__ == "__main__":
    main()


WEEK 11 FUNCTION 3 — CLUSTERING PRE-CHECKS
Points (n): 25
Frac out-of-bounds in X_raw (before clip): 0.040

--- TUNING STAGE 1 (W11) ---
epochs=450, kept=5/10
best CV-MSE so far: 0.027436
best config so far: {'hidden': (64, 32), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 1e-06, 'n_mc_samples': 128, 'epochs': 450}

--- TUNING STAGE 2 (W11) ---
epochs=1050, kept=4/5
best CV-MSE so far: 0.027436
best config so far: {'hidden': (64, 32), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 1e-06, 'n_mc_samples': 128, 'epochs': 450}


C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai


WEEK 11 FUNCTION 3 — NEXT POINT (6 DECIMALS)
Surrogate: ensemble(mc_dropout) + robust y scaling
Best CV-MSE (lower is better): 0.027436
Best tuned hyperparameters:
  hidden: (64, 32)
  dropout: 0.2
  lr: 0.0007
  weight_decay: 1e-06
  n_mc_samples: 128
  epochs: 450
Ensemble seeds: [0, 11, 29]

CURRENT BEST OBSERVED
x_best = [0.403756, 0.381706, 0.489738], y_best = -0.009136

RECOMMENDED NEXT POINT (rounded to 6 decimals)
x_next     = [0.972960, 0.967101, 0.487866]
μ(x_next)  = -0.167035
σ(x_next)  = 0.070227

CLUSTERING SUMMARY (TRANSPARENCY)
k (silhouette-selected) = 2 | silhouette=0.353
top_clusters (ranked by best/mean y) = [1, 0]
x_next targets cluster = 0

TRANSPARENT SCORE DECOMPOSITION @ x_next
score       = 0.624852
comp_ei     = 0.621270  (EI_norm=0.955800, xi=0.002600)
comp_ucb    = 0.000000  (UCB_impr=0.000000, beta=0.9985)
cluster_bonus= 0.012612
comp_repulse= 0.009031  (dmin=0.132347)
comp_bound  = 0.000000  (bound_close=0.027040, margin=0.006)

INTERPRETABILITY: LOCAL S